# 🎯 Model Evaluation & Error Analysis

This notebook helps quantify where your current `0.86` model is still failing.
As you noted, *nicer CoT* isn't enough anymore; we need to pinpoint exactly which puzzle families are solved and which are lacking.

The dataset is segmented into:
- Bit manipulation
- Gravity
- Unit conversion
- Roman numerals
- Text ciphers
- Equation transforms

**Workflow**:
1. We run your `0.86` model locally over your validation or training split.
2. We log per-category accuracy.
3. We deep-dive into `Bit manipulation` by segmenting by `puzzle id` prefixes to isolate families with `< 1.0` accuracy.

In [ ]:
# Set up environment (Adapted from the official NVIDIA Evaluation Metric)
import subprocess
import sys
import os

IN_KAGGLE = os.path.exists('/kaggle')

if IN_KAGGLE:
    commands = [
        'uv pip uninstall torch torchvision torchaudio',
        'tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp',
        'chmod +x /tmp/triton/backends/nvidia/bin/ptxas',
        'chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell',
    ]
    for cmd in commands:
        print(f'Running: {cmd}')
        subprocess.run(cmd, shell=True, check=False)
    sys.path.insert(0, '/tmp')


In [ ]:
import glob
import math
import multiprocessing
import re
import time
from pathlib import Path

import pandas as pd
from tqdm import tqdm

# Configuration - Update paths according to your local setup
if IN_KAGGLE:
    import kagglehub
    MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
    DATA_PATH = Path('./data') 
    LORA_PATH = None
else:
    MODEL_PATH = 'nvidia/nemotron-3-nano-30b-a3b-bf16' # << UPDATE: Base model path (or HF repo)
    DATA_PATH = Path('./data')       # << UPDATE: Path to your train/eval split directory
    LORA_PATH = './models/nemotron_adapter' # << UPDATE: Path to your adapter directory


## 1. NVIDIA Evaluation Metric Utilities
These are the exact functions from the official metric for answer extraction and verification.

In [ ]:
def cache_model(path, exts=('.bin', '.pt', '.safetensors'), num_workers=None, chunk_mb=256):
    from concurrent.futures import ThreadPoolExecutor, as_completed
    def warmup_file(fpath):
        chunk_size = chunk_mb * 1024 * 1024
        total = 0
        try:
            with open(fpath, 'rb') as f:
                while True:
                    data = f.read(chunk_size)
                    if not data: break
                    total += len(data)
        except Exception as e:
            print(f'Error reading {fpath}: {e}')
        return fpath, total

    path = Path(path)
    files = []
    if path.is_dir():
        files = [p for p in path.rglob('*') if p.is_file() and str(p).endswith(exts)]
        files.sort()
    else:
        files = [path] if path.exists() else []

    if not files: return 0

    num_workers = min(multiprocessing.cpu_count(), 8) if num_workers is None else num_workers
    print(f'[cache_model] {len(files)} file(s), {num_workers} worker(s)')
    t0 = time.time()
    total_bytes = 0
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        futures = {pool.submit(warmup_file, f): f for f in files}
        for i, fut in enumerate(as_completed(futures), 1):
            fpath, n = fut.result()
            total_bytes += n
    elapsed = time.time() - t0
    gb = total_bytes / 1024**3
    print(f'[cache_model] total read ≈ {gb:.2f} GB in {elapsed:.2f}s')
    return total_bytes

def extract_final_answer(text):
    if text is None: return 'NOT_FOUND'
    boxed_starts = list(re.finditer(r'\\boxed\{', text))
    matches = []
    for i, m in enumerate(boxed_starts):
        start = m.end()
        end = boxed_starts[i + 1].start() if i + 1 < len(boxed_starts) else len(text)
        segment = text[start:end]
        last_brace = segment.rfind('}')
        matches.append(segment[:last_brace] if last_brace != -1 else segment)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        return non_empty[-1] if non_empty else matches[-1].strip()
    patterns = [
        r'The final answer is:\s*([^\n]+)',
        r'Final answer is:\s*([^\n]+)',
        r'Final answer\s*[:：]\s*([^\n]+)',
        r'final answer\s*[:：]\s*([^\n]+)',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches: return matches[-1].strip()
    matches = re.findall(r'-?\d+(?:\.\d+)?', text)
    if matches: return matches[-1]
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else 'NOT_FOUND'

def verify(stored_answer, predicted):
    stored_answer, predicted = str(stored_answer).strip(), str(predicted).strip()
    if re.fullmatch(r'[01]+', stored_answer):
        return predicted.lower() == stored_answer.lower()
    try:
        return math.isclose(float(stored_answer), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
    except:
        return predicted.lower() == stored_answer.lower()


## 2. Categorization Logic
This splits the tasks into families to perform the targeted accuracy check.

In [ ]:
def categorize_task(row):
    """
    Assign a category and sub-category to each puzzle based on ID or prompt context.
    """
    task_id = str(row.get('id', row.get('row_id', ''))).lower()
    prompt = str(row.get('prompt', row.get('question', ''))).lower()
    
    category = 'Other'
    sub_category = 'Other'
    
    if 'bit' in task_id or 'binary' in prompt or 'gf2' in task_id:
        category = 'Bit manipulation'
        # Extract prefix from ID (e.g. 'bit_manip_gf2_01' -> 'bit_manip_gf2')
        parts = task_id.split('_')
        sub_category = "_".join(parts[:-1]) if len(parts) > 1 else task_id
    elif 'grav' in task_id or 'physics' in task_id:
        category = 'Gravity'
    elif 'unit' in task_id or 'convert' in prompt:
        category = 'Unit conversion'
    elif 'roman' in task_id or 'roman' in prompt:
        category = 'Roman numerals'
    elif 'cipher' in task_id or 'crypto' in task_id:
        category = 'Text ciphers'
    elif 'eq' in task_id or 'math' in task_id:
        category = 'Equation transforms'
        
    return pd.Series({'category': category, 'sub_category': sub_category})

def prepare_dataset(csv_path):
    df = pd.read_csv(csv_path)
    cat_df = df.apply(categorize_task, axis=1)
    df = pd.concat([df, cat_df], axis=1)
    return df


## 3. Generate Predictions & Evaluate

In [ ]:
def evaluate_model_by_category(
    eval_df: pd.DataFrame,
    lora_path: str = None,
    max_lora_rank: int = 32,
    max_tokens: int = 3584,
    top_p: float = 1.0,
    temperature: float = 0.0, # Temperature 0 for stable eval
    max_num_seqs: int = 128,
    gpu_memory_utilization: float = 0.85,
    max_model_len: int = 4096,
) -> pd.DataFrame:
    
    # Cache Model
    cache_model(MODEL_PATH, num_workers=16, chunk_mb=1024)

    os.environ['TRANSFORMERS_NO_TF'] = '1'
    os.environ['TRANSFORMERS_NO_FLAX'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    # os.environ['CUDA_VISIBLE_DEVICES'] = '0'

    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest

    kwargs = {
        'model': str(MODEL_PATH),
        'tensor_parallel_size': 1,
        'max_num_seqs': max_num_seqs,
        'gpu_memory_utilization': gpu_memory_utilization,
        'dtype': 'auto',
        'max_model_len': max_model_len,
        'trust_remote_code': True,
        'enable_prefix_caching': True,
        'enable_chunked_prefill': True,
    }
    
    if lora_path:
        kwargs['enable_lora'] = True
        kwargs['max_lora_rank'] = max_lora_rank

    llm = LLM(**kwargs)

    sampling_params = SamplingParams(
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )

    tokenizer = llm.get_tokenizer()
    prompts = []
    
    for item in eval_df.itertuples(index=False):
        user_content = (
            item.prompt
            + '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
        )
        try:
            prompt = tokenizer.apply_chat_template(
                [{'role': 'user', 'content': user_content}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
            )
        except Exception:
            prompt = user_content
        prompts.append(prompt)

    lora_request = LoRARequest('adapter', 1, lora_path) if lora_path else None
    
    outputs = llm.generate(
        prompts,
        sampling_params=sampling_params,
        lora_request=lora_request,
    )

    results = []
    for item, output in zip(eval_df.itertuples(index=False), outputs):
        raw_text = output.outputs[0].text
        extracted_answer = extract_final_answer(raw_text)
        ground_truth = str(item.answer)
        is_correct = verify(ground_truth, extracted_answer)
        
        results.append({
            'id': item.id if hasattr(item, 'id') else (item.row_id if hasattr(item, 'row_id') else None),
            'category': item.category,
            'sub_category': item.sub_category,
            'ground_truth': ground_truth,
            'prediction': extracted_answer,
            'is_correct': is_correct,
            'raw_output': raw_text
        })

    return pd.DataFrame(results)


## 4. Run & Analyze
Uncomment and run below. Point to your local `train.csv` (or validation split).

In [ ]:
# df_train = prepare_dataset(DATA_PATH / 'train.csv')
# print(f"Loaded {len(df_train)} rows for evaluation.")

# # RUN EVALUATION
# df_results = evaluate_model_by_category(df_train, lora_path=LORA_PATH) 

# # Save results for offline analysis
# df_results.to_csv('evaluation_results_086.csv', index=False)

# print("\n===========================")
# print("=== Overall Accuracy ===")
# print(f"{df_results['is_correct'].mean():.4f}")
# print("===========================\n")

# print("=== Accuracy by Category ===")
# cat_acc = df_results.groupby('category')['is_correct'].agg(['mean', 'count'])
# print(cat_acc)

# print("\n=== Accuracy for Bit Manipulation Families ===")
# bit_df = df_results[df_results['category'] == 'Bit manipulation']
# if not bit_df.empty:
#     bit_acc = bit_df.groupby('sub_category')['is_correct'].agg(['mean', 'count']).sort_values('mean')
#     print(bit_acc)
